# BERTScore Evaluation

This notebook evaluates model predictions against the gold solutions in `Evaluation/solutions.csv` using only BERTScore.

Expected files:
- Gold answers: `Evaluation/solutions.csv`
- Predictions: CSV files in `Results/` with columns `id`, `answer`


In [1]:
import csv
import importlib.util
from pathlib import Path

import pandas as pd

if importlib.util.find_spec("bert_score") is None:
    raise ModuleNotFoundError(
        "`bert_score` is not installed. Run `%pip install bert-score pandas` in a notebook cell, then restart the kernel."
    )

from bert_score import score as bertscore_score

ROOT = Path.cwd()
if not (ROOT / "Results").exists() and (ROOT.parent / "Results").exists():
    ROOT = ROOT.parent

EVALUATION_DIR = ROOT / "Evaluation"
RESULTS_DIR = ROOT / "Results"
GOLD_PATH = EVALUATION_DIR / "solutions.csv"

MODEL_FILES = {
    "qwen_1_5b_base": RESULTS_DIR / "inference_qwen_1_5b.csv",
    "qwen_1_5b_finetuned": RESULTS_DIR / "finetune_qwen_1_5b.csv",
    "qwen_1_5b_rag": RESULTS_DIR / "RAG_qwen_1_5b.csv",
    "qwen_14b": RESULTS_DIR / "qwen_14b_test.csv",
}

LANGUAGE = "de"
GOLD_PATH


/Users/lucas/Documents/Data Applications/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/Users/lucas/Downloads/wu-llms-ss26-submission_LH/Submission_Lucas_Harrich/Evaluation/solutions.csv')

In [2]:
COLUMN_RENAME_MAP = {
    "correct_answer": "answer",
    "gold_answer": "answer",
    "reference_answer": "answer",
    "prompt": "question",
}


def normalize_columns(columns) -> list[str]:
    normalized_columns = [str(column).strip().lower() for column in columns]
    return [COLUMN_RENAME_MAP.get(column, column) for column in normalized_columns]


def detect_delimiter(path: Path) -> str:
    sample = path.read_text(encoding="utf-8-sig")[:8192]
    candidate_delimiters = []

    try:
        candidate_delimiters.append(csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter)
    except csv.Error:
        pass

    for delimiter in [",", ";", "\t", "|"]:
        if delimiter not in candidate_delimiters:
            candidate_delimiters.append(delimiter)

    required_columns = {"id", "answer"}
    for delimiter in candidate_delimiters:
        try:
            preview_df = pd.read_csv(path, sep=delimiter, encoding="utf-8-sig", nrows=5)
        except Exception:
            continue

        if required_columns.issubset(set(normalize_columns(preview_df.columns))):
            return delimiter

    raise ValueError(f"Could not determine delimiter for {path.name}")


def read_answers_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    delimiter = detect_delimiter(path)
    df = pd.read_csv(path, sep=delimiter, encoding="utf-8-sig")
    df.columns = normalize_columns(df.columns)

    required_columns = {"id", "answer"}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"{path.name} is missing required columns: {sorted(missing_columns)}")

    df["id"] = df["id"].astype(str).str.strip()
    df["answer"] = df["answer"].fillna("").astype(str).str.strip()

    if "question" in df.columns:
        df["question"] = df["question"].fillna("").astype(str).str.strip()

    return df


gold_df = read_answers_csv(GOLD_PATH)
gold_df = gold_df[[column for column in ["id", "question", "answer"] if column in gold_df.columns]]
gold_df = gold_df.drop_duplicates(subset=["id"], keep="first")

print(f"Loaded {len(gold_df)} gold answers from {GOLD_PATH.name}")
gold_df.head()


Loaded 643 gold answers from solutions.csv


,id,question,answer
0,CORP-TAX-001,Was ist die steuerliche Bemessungsgrundlage fü...,"Das Einkommen, das der unbeschränkt Steuerpfli..."
1,CORP-TAX-002,"Welche steuerlichen Konsequenzen hat es, wenn ...",Die nicht verrechneten Zinsen stellen eine ver...
2,CORP-TAX-003,"Welche Körperschaften sind verpflichtet, sämtl...","Steuerpflichtige, die aufgrund ihrer Rechtsfor..."
3,CORP-TAX-004,Wie wird eine Dividende einer österreichischen...,(a) Tochter: Die Ausschüttung ist Einkommensve...
4,CORP-TAX-005,Was unterscheidet die steuerliche Behandlung e...,Im Ergebnis unterscheiden sie sich nicht. Beid...


In [3]:
def evaluate_model(model_name: str, prediction_path: Path, gold_answers: pd.DataFrame):
    prediction_df = read_answers_csv(prediction_path)
    prediction_df = prediction_df[["id", "answer"]].drop_duplicates(subset=["id"], keep="first")

    merged_df = gold_answers.merge(
        prediction_df.rename(columns={"answer": "prediction"}),
        on="id",
        how="inner",
    ).rename(columns={"answer": "gold_answer"})

    if merged_df.empty:
        raise ValueError(f"No overlapping ids between gold file and {prediction_path.name}")

    _, _, f1_scores = bertscore_score(
        merged_df["prediction"].tolist(),
        merged_df["gold_answer"].tolist(),
        lang=LANGUAGE,
        verbose=False,
    )

    merged_df["bertscore_f1"] = f1_scores.tolist()

    summary_row = {
        "model": model_name,
        "prediction_file": prediction_path.name,
        "rows_scored": len(merged_df),
        "gold_rows": len(gold_answers),
        "bertscore_f1": float(merged_df["bertscore_f1"].mean()),
    }
    return merged_df, summary_row


details_by_model = {}
summary_rows = []

for model_name, prediction_path in MODEL_FILES.items():
    if not prediction_path.exists():
        print(f"Skipping {model_name}: {prediction_path.name} not found")
        continue

    details_df, summary_row = evaluate_model(model_name, prediction_path, gold_df)
    details_by_model[model_name] = details_df.sort_values("bertscore_f1")
    summary_rows.append(summary_row)

summary_df = pd.DataFrame(summary_rows).sort_values("bertscore_f1", ascending=False)
summary_df["bertscore_f1"] = summary_df["bertscore_f1"].round(4)
summary_df


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2638.97it/s]
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6062.63it/s]
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------

,model,prediction_file,rows_scored,gold_rows,bertscore_f1
3,qwen_14b,qwen_14b_test.csv,641,643,0.7078
1,qwen_1_5b_finetuned,finetune_qwen_1_5b.csv,641,643,0.6857
0,qwen_1_5b_base,inference_qwen_1_5b.csv,641,643,0.6845
2,qwen_1_5b_rag,RAG_qwen_1_5b.csv,641,643,0.6784


In [4]:
summary_output_path = EVALUATION_DIR / "bertscore_summary.csv"
summary_df.to_csv(summary_output_path, index=False)

for model_name, details_df in details_by_model.items():
    details_output_path = EVALUATION_DIR / f"{model_name}_bertscore_details.csv"
    details_df.to_csv(details_output_path, index=False)

print(f"Saved summary to {summary_output_path}")
for model_name in details_by_model:
    print(f"Saved details for {model_name} to {EVALUATION_DIR / (model_name + '_bertscore_details.csv')}")

summary_df


Saved summary to /Users/lucas/Downloads/wu-llms-ss26-submission_LH/Submission_Lucas_Harrich/Evaluation/bertscore_summary.csv
Saved details for qwen_1_5b_base to /Users/lucas/Downloads/wu-llms-ss26-submission_LH/Submission_Lucas_Harrich/Evaluation/qwen_1_5b_base_bertscore_details.csv
Saved details for qwen_1_5b_finetuned to /Users/lucas/Downloads/wu-llms-ss26-submission_LH/Submission_Lucas_Harrich/Evaluation/qwen_1_5b_finetuned_bertscore_details.csv
Saved details for qwen_1_5b_rag to /Users/lucas/Downloads/wu-llms-ss26-submission_LH/Submission_Lucas_Harrich/Evaluation/qwen_1_5b_rag_bertscore_details.csv
Saved details for qwen_14b to /Users/lucas/Downloads/wu-llms-ss26-submission_LH/Submission_Lucas_Harrich/Evaluation/qwen_14b_bertscore_details.csv


,model,prediction_file,rows_scored,gold_rows,bertscore_f1
3,qwen_14b,qwen_14b_test.csv,641,643,0.7078
1,qwen_1_5b_finetuned,finetune_qwen_1_5b.csv,641,643,0.6857
0,qwen_1_5b_base,inference_qwen_1_5b.csv,641,643,0.6845
2,qwen_1_5b_rag,RAG_qwen_1_5b.csv,641,643,0.6784
